# M8 — Học đa nhiệm trên CIFAR-10

**Phân loại siêu lớp kết hợp dự đoán góc xoay ảnh tự giám sát**

| | |
|---|---|
| Sinh viên | Đinh Xuân Huy |
| MSSV | 23110102 |
| Học phần | Trí tuệ nhân tạo cho IoT |
| Mã đề tài | M8 |

---

Notebook này trình bày **toàn bộ quy trình** của đề tài, theo đúng thứ tự của báo cáo:

1. Môi trường và kiểm tra kỹ thuật
2. Dữ liệu: ánh xạ siêu lớp, chia split, ngân sách nhãn, kiểm tra chống rò rỉ nhãn
3. Kiến trúc mô hình và đếm tham số
4. Hàm mất mát và sinh nhãn góc xoay
5. Huấn luyện một lượt (E1 / E2-L / E2)
6. Khối thực nghiệm chính: 3 cấu hình × 3 mức nhãn × 3 lần lặp
7. Chỉ số, ma trận nhầm lẫn, chênh lệch ghép cặp
8. Hành vi hội tụ và chi phí suy luận
9. Kết luận tự động từ kết quả
10. Mở rộng contrastive: NT-Xent và SupCon (E3–E6)
11. Ghi chú về tái lập và giới hạn

> **Lưu ý về tính trung thực:** mọi con số trong notebook đều đọc từ tệp kết quả thật
> (`results/`). Ô nào cần kết quả huấn luyện sẽ tự phát hiện nếu chưa có và nói rõ,
> chứ không sinh số liệu giả.

In [ ]:
# === 0. Thiết lập đường dẫn và import ===
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if (ROOT / 'src' / 'm8').exists():
    pass
elif (ROOT.parent / 'src' / 'm8').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
print('PROJECT ROOT =', ROOT)

from m8 import config as C
from m8.data import (SuperclassDataset, RotationViewDataset, audit_benchmark,
                     build_benchmark, get_cifar10, labeled_train_transform,
                     eval_transform, make_loader, make_run_loaders)
from m8.models import build_model, count_parameters, model_summary
from m8.evaluate import ConfusionMatrix, evaluate_predictions, macro_f1
from m8.train import train_one_run, make_optimizer, make_scheduler, validate
from m8.experiments import load_all_runs, summarize_runs, _init_state_for

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', DEVICE)

## 1. Môi trường và kiểm tra kỹ thuật

Thông tin môi trường được đọc từ `results/environment.json` do
`scripts/run_inference_benchmark.py` sinh ra, và kết quả kiểm tra kỹ thuật từ
`results/self_test.json` do `scripts/self_test.py` sinh ra.

Hai con số quan trọng cần khớp với thiết kế trong đề cương:

* **11 169 858** tham số khi suy luận (encoder + Superclass Head)
* **11 171 910** tham số khi huấn luyện (thêm Rotation Head 2 052 tham số)

In [ ]:
self_test = json.loads((ROOT / 'results' / 'self_test.json').read_text(encoding='utf-8'))
env = self_test['environment']

env_df = pd.DataFrame({'Thuộc tính': list(env.keys()), 'Giá trị': [str(v) for v in env.values()]})
display(env_df)

EXPECTED_INFERENCE = 11_169_858
EXPECTED_TRAINING = 11_171_910
params = self_test['parameters']
print('tham số suy luận :', f"{params['inference_total']:,}",
      'kỳ vọng', f'{EXPECTED_INFERENCE:,}',
      '->', 'ĐẠT' if params['inference_total'] == EXPECTED_INFERENCE else 'KHÔNG ĐẠT')
print('tham số huấn luyện:', f"{params['training_total']:,}",
      'kỳ vọng', f'{EXPECTED_TRAINING:,}',
      '->', 'ĐẠT' if params['training_total'] == EXPECTED_TRAINING else 'KHÔNG ĐẠT')
print('Rotation Head     :', f"{params['rotation_head']:,} tham số")
print('FP32 suy luận     :', f"{params['inference_fp32_mib']:.2f} MiB")
print()
print('Kết quả tự kiểm tra:', 'ĐẠT' if self_test['passed'] else 'KHÔNG ĐẠT')

In [ ]:
# Chuỗi kích thước tensor (bảng 4 của báo cáo)
probe = build_model(with_rotation_head=True, seed=0, device='cpu')
shapes = pd.DataFrame(probe.encoder.feature_shapes(), columns=['Thành phần', 'Kích thước đầu ra'])
shapes['Kích thước đầu ra'] = shapes['Kích thước đầu ra'].apply(lambda t: ' × '.join(map(str, t)))
display(shapes)
print(json.dumps(model_summary(probe), ensure_ascii=False, indent=2))

## 2. Dữ liệu: ánh xạ siêu lớp và chia split

CIFAR-10 có 60 000 ảnh $32\times32$: 50 000 train và 10 000 test. Hai siêu lớp
`Animal` và `Vehicle` là **phép gộp của đề tài**, không phải nhãn chính thức của CIFAR-10.

* `Animal` = {bird, cat, deer, dog, frog, horse} → tỷ lệ 60 %
* `Vehicle` = {airplane, automobile, ship, truck} → tỷ lệ 40 %

Vì tỷ lệ 60/40, một bộ phân loại luôn đoán `Animal` đã đạt Accuracy 60 %. Do đó
**Macro-F1 là chỉ số chính**.

In [ ]:
mapping = pd.DataFrame({
    'Nhãn mới': [0, 1],
    'Siêu lớp': ['Animal', 'Vehicle'],
    'Lớp gốc': [', '.join(C.CLASS_NAMES[c] for c in C.ANIMAL_CLASSES),
                ', '.join(C.CLASS_NAMES[c] for c in C.VEHICLE_CLASSES)],
    'Số lớp': [len(C.ANIMAL_CLASSES), len(C.VEHICLE_CLASSES)],
})
display(mapping)

budget = pd.DataFrame([
    {'Tỷ lệ': f"{int(round(f*100))}%", 'D_L': b['total'], 'Animal': b['animal'],
     'Vehicle': b['vehicle'], 'D_U (ẩn nhãn)': b['unlabeled'], 'Tổng D_train': 45000}
    for f, b in sorted(C.LABEL_BUDGET.items())
])
display(budget)

In [ ]:
# === Tải CIFAR-10 và dựng phép chia benchmark (đóng băng seed 2026) ===
train_base, test_base = get_cifar10(download=True)
print('trainset', len(train_base), '| testset', len(test_base))

bench = build_benchmark(train_base.targets)
audit = audit_benchmark(bench, train_base.targets)
print('D_train', bench.train_idx.size, '| D_val', bench.val_idx.size, '| D_test', 10_000)

# Kiểm tra các tính chất quan trọng: lồng nhau, rời nhau, không rò rỉ sang validation
for key, info in audit['label_budget'].items():
    assert info['disjoint'] and info['no_val_leak'], key
for seed_info in audit['nesting'].values():
    assert seed_info['nested']
assert audit['no_overlap_train_val']
print('\nKiểm tra chống rò rỉ nhãn: ĐẠT (D_L ∩ D_U = ∅, (D_L ∪ D_U) ∩ D_val = ∅, tập lồng nhau)')
print('Tỷ lệ Animal trong D_train:', f"{audit['train_animal_share']:.3f}",
      '| trong D_val:', f"{audit['val_animal_share']:.3f}")

In [ ]:
# === Kiểm tra hai nhánh dùng view khác nhau, và nhãn k khớp chính xác với torch.rot90 ===
idx = bench.dl(42, 0.10)[:256]
rot_ds = RotationViewDataset(train_base, idx)
batch = [rot_ds[i] for i in range(64)]
X = torch.stack([b[0] for b in batch])
K = torch.tensor([b[1] for b in batch])
print('shape batch xoay:', tuple(X.shape))
print('phân bố k:', torch.bincount(K, minlength=4).tolist())

# dựng lại ảnh chuẩn hóa từ ảnh gốc và so với tensor đã xoay
ok = True
for i in range(16):
    raw = np.asarray(train_base[int(idx[i])][0], dtype='float32') / 255.0
    raw = torch.from_numpy(raw).permute(2, 0, 1)
    norm = (raw - 0.5) / 0.5
    expect = torch.rot90(norm, k=int(K[i]), dims=(1, 2))
    ok &= bool(torch.allclose(expect, X[i], atol=1e-5))
print('nhãn k khớp torch.rot90:', ok)

# minh họa bốn phép xoay của cùng một ảnh
fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
img = np.asarray(train_base[int(idx[0])][0])
for k, ax in enumerate(axes):
    ax.imshow(np.rot90(img, k=k))
    ax.set_title(f'k = {k}')
    ax.axis('off')
fig.suptitle('Bốn phép xoay 90k độ dùng sinh nhãn cho nhiệm vụ phụ (ngược chiều kim đồng hồ)')
plt.tight_layout(); plt.show()

## 3. Kiến trúc và hàm mất mát

Backbone là ResNet-18 khởi tạo ngẫu nhiên (không tiền huấn luyện), điều chỉnh cho ảnh
$32\times32$: `Conv2d(3, 64, 3, stride=1, padding=1, bias=False)`, bỏ MaxPool đầu mạng,
giữ bốn nhóm residual × 2 BasicBlock.

Hai hàm mất mát được lấy trung bình **riêng**:

$$L_{\mathrm{sup}}=\frac{1}{b_L}\sum_{i=1}^{b_L}\mathrm{CE}(a_{\mathrm{sup},i},y_i),\qquad
L_{\mathrm{rot}}=\frac{1}{b_R}\sum_{j=1}^{b_R}\mathrm{CE}(a_{\mathrm{rot},j},k_j),\qquad
L=L_{\mathrm{sup}}+\lambda_{\mathrm{rot}}L_{\mathrm{rot}}$$

In [ ]:
def multi_task_loss(out_sup, y_sup, out_rot, y_rot, lam=C.LAMBDA_ROT):
    """Trả về (loss_sup, loss_rot, loss_tổng) — hai mất mát trung bình riêng."""
    crit = torch.nn.CrossEntropyLoss()
    l_sup = crit(out_sup, y_sup)
    if out_rot is None:
        return l_sup, None, l_sup
    l_rot = crit(out_rot, y_rot)
    return l_sup, l_rot, l_sup + lam * l_rot


x_sup = torch.randn(8, 3, 32, 32)
y_sup = torch.randint(0, 2, (8,))
x_rot = torch.randn(8, 3, 32, 32)
y_rot = torch.randint(0, 4, (8,))

# E1: KHÔNG có head xoay, KHÔNG chạy nhánh xoay kể cả forward
m_e1 = build_model(with_rotation_head=False, seed=42)
l_sup, l_rot, l_tot = multi_task_loss(m_e1(x_sup), y_sup, None, None)
print('E1   :', f'L_sup={l_sup.item():.4f}', '| có Rotation Head:', m_e1.rotation_head is not None)

# E2-L / E2: ghép hai minibatch theo chiều batch rồi tách đặc trưng về đúng head
m_m8 = build_model(with_rotation_head=True, seed=42)
out_sup, out_rot = m_m8.forward_joined(x_sup, x_rot)
l_sup, l_rot, l_tot = multi_task_loss(out_sup, y_sup, out_rot, y_rot)
print('M8   :', f'L_sup={l_sup.item():.4f}', f'L_rot={l_rot.item():.4f}',
      f'L_tổng={l_tot.item():.4f}', f'(lambda_rot={C.LAMBDA_ROT})')

# Kiểm tra BatchNorm dùng chung: đưa ảnh xoay qua encoder làm đổi thống kê chạy
m_m8.train(); m_e1.train()
before = m_m8.encoder.bn1.running_mean.clone()
m_m8.forward_joined(x_sup, x_rot)
after = m_m8.encoder.bn1.running_mean
print('BatchNorm dùng chung cho hai nhánh -> thống kê chạy thay đổi:',
      not torch.allclose(before, after), f'(|Δ| = {(after-before).abs().mean():.3e})')

## 4. Ba cấu hình đối chứng

| ID | Cấu hình | Ảnh cho phân loại | Ảnh cho nhiệm vụ xoay |
|---|---|---|---|
| E1 | Baseline chỉ học siêu lớp | $D_L$ | không có |
| E2-L | Thêm nhánh xoay, cùng tập nguồn | $D_L$ | chỉ $D_L$ |
| E2 | Thêm nhánh xoay, mở rộng nguồn ảnh | $D_L$ | $D_{train} = D_L \cup D_U$ |

Phép so sánh **E2-L vs E1** cô lập tác dụng của việc *thêm nhiệm vụ phụ*;
**E2 vs E2-L** cô lập tác dụng của việc *mở rộng nguồn ảnh*.

In [ ]:
configs = pd.DataFrame([
    {'ID': 'E1', 'Cấu hình': 'Baseline chỉ học siêu lớp', 'Ảnh phân loại': 'D_L',
     'Ảnh cho rotation': 'không có'},
    {'ID': 'E2-L', 'Cấu hình': 'M8 dùng cùng tập ảnh nguồn', 'Ảnh phân loại': 'D_L',
     'Ảnh cho rotation': 'chỉ D_L'},
    {'ID': 'E2', 'Cấu hình': 'M8 có thêm ảnh ẩn nhãn', 'Ảnh phân loại': 'D_L',
     'Ảnh cho rotation': 'D_train = D_L ∪ D_U'},
])
display(configs)

# Số bước cập nhật mỗi epoch và số lượt ảnh phụ thuộc cấu hình × mức nhãn
rows = []
for cfg in C.MAIN_CONFIGS:
    for f in C.LABEL_FRACTIONS:
        n_lab = C.LABEL_BUDGET[f]['total']
        steps = int(np.ceil(n_lab / C.BATCH_SIZE))
        rot_images = 0 if cfg == 'E1' else steps * C.BATCH_SIZE
        rows.append({'Cấu hình': cfg, 'Nhãn': f'{int(f*100)}%', 'bước/epoch': steps,
                     'lượt ảnh phân loại/epoch': n_lab,
                     'lượt ảnh rotation/epoch': rot_images})
display(pd.DataFrame(rows))

## 5. Huấn luyện một lượt

Hàm `train_one_run` trong `src/m8/train.py` thực hiện đúng giao thức:

* một epoch = một lượt duyệt $D_L$;
* AdamW, lr $3\times10^{-4}$, weight decay $10^{-4}$, betas (0.9, 0.999);
* CosineAnnealingLR với $T_{max}=100$, cập nhật sau mỗi epoch;
* $B_R$ lấy bằng một **bộ sinh số ngẫu nhiên độc lập** để không dịch chuyển chuỗi
  augmentation của nhánh phân loại;
* chọn checkpoint theo Macro-F1 validation cao nhất (hòa → loss thấp hơn → epoch sớm hơn);
* **không** dùng Early Stopping trong bảng kết quả chính.

> Ô dưới chạy 2 epoch như một *minh họa cơ chế*, không phải kết quả báo cáo. Kết quả thật
> nằm trong `results/checkpoints/` do `scripts/run_experiments.py` sinh ra.

In [ ]:
spec = C.RunSpec(config='E2', label_fraction=0.10, seed=42, epochs=2)
loaders = make_run_loaders(train_base, test_base, bench, spec.seed, spec.label_fraction,
                           spec.rotation_source)
init_state = _init_state_for(spec)
demo, demo_model = train_one_run(spec, loaders, DEVICE, init_state=init_state,
                                 amp=torch.cuda.is_available(), verbose=True)
print('\nDemo xong. val Macro-F1 =', round(demo.best_val['macro_f1'], 4),
      '| test Macro-F1 =', round(demo.test['macro_f1'], 4))
print('Lưu ý: 2 epoch là quá ít để kết luận; bảng kết quả chính dùng kết quả 100 epoch.')

## 6. Khối thực nghiệm chính

3 cấu hình × 3 mức nhãn × 3 lần lặp = **27 lượt huấn luyện**, mỗi lượt 100 epoch.

Nếu máy đã chạy `scripts/run_experiments.py`, các tệp kết quả đã có sẵn. Ngược lại, ô
dưới đây sẽ chạy bổ sung những lượt còn thiếu (có thể rất lâu).

In [ ]:
specs = C.main_run_specs()
done = {p.stem for p in (ROOT / 'results' / 'checkpoints').glob('*.json')}
missing = [s for s in specs if s.tag not in done]
print(f'Tổng {len(specs)} lượt | đã có {len(specs) - len(missing)} | còn thiếu {len(missing)}')
for s in missing:
    print('  thiếu:', s.tag)

RUN_MISSING = False   # đổi thành True nếu muốn notebook tự chạy bổ sung
if RUN_MISSING and missing:
    from m8.experiments import run_suite
    print('Bắt đầu chạy bổ sung...')
    run_suite(specs=missing, verbose=True)
    done = {p.stem for p in (ROOT / 'results' / 'checkpoints').glob('*.json')}

## 7. Kết quả: Macro-F1, Accuracy, chỉ số từng lớp

Tất cả số liệu dưới đây đọc trực tiếp từ `results/checkpoints/*.json`.

In [ ]:
runs = load_all_runs(ROOT / 'results' / 'checkpoints')
print('số lượt có kết quả:', len(runs))
if not runs:
    print('Chưa có kết quả — hãy chạy scripts/run_experiments.py trước.')

summary = summarize_runs(runs) if runs else {'runs': [], 'aggregate': {}, 'paired_delta': {}}
flat = pd.DataFrame(summary['runs'])
if not flat.empty:
    show_cols = ['tag', 'config', 'label_pct', 'seed', 'best_epoch', 'val_macro_f1',
                 'test_macro_f1', 'test_accuracy', 'train_seconds']
    display(flat[show_cols].sort_values(['label_pct', 'config', 'seed']).reset_index(drop=True))

In [ ]:
# Bảng tổng hợp: Macro-F1 / Accuracy trên D_test, mean ± std qua 3 lần lặp
if summary['aggregate']:
    rows = []
    for cfg in C.MAIN_CONFIGS:
        for pct in sorted({r['label_pct'] for r in summary['runs']}):
            a = summary['aggregate'].get(f'{cfg}_{pct}')
            d = summary['paired_delta'].get(f'{cfg}_minus_E1_{pct}')
            if not a:
                continue
            rows.append({
                'Cấu hình': cfg,
                'Nhãn': f'{pct}%',
                'Test Macro-F1': f"{a['test_macro_f1']['mean']:.4f} ± {a['test_macro_f1']['std']:.4f}",
                'Test Accuracy': f"{a['test_accuracy']['mean']:.4f} ± {a['test_accuracy']['std']:.4f}",
                'Δ so với E1 (điểm %)': ('—' if cfg == 'E1' or not d else f"{d['mean']:+.2f}"),
                'Giây/lượt': f"{a['train_seconds']['mean']:.0f}",
            })
    tbl = pd.DataFrame(rows)
    display(tbl)
else:
    print('Chưa có kết quả để tổng hợp.')

In [ ]:
# Chỉ số từng siêu lớp tại mức 10% nhãn
if summary['runs']:
    rows = []
    for cfg in C.MAIN_CONFIGS:
        sel = [r for r in summary['runs'] if r['config'] == cfg and r['label_pct'] == 10]
        if not sel:
            continue
        rows.append({
            'Cấu hình': cfg,
            'P(Animal)': np.mean([r['test_precision_animal'] for r in sel]),
            'R(Animal)': np.mean([r['test_recall_animal'] for r in sel]),
            'F1(Animal)': np.mean([r['test_f1_animal'] for r in sel]),
            'P(Vehicle)': np.mean([r['test_precision_vehicle'] for r in sel]),
            'R(Vehicle)': np.mean([r['test_recall_vehicle'] for r in sel]),
            'F1(Vehicle)': np.mean([r['test_f1_vehicle'] for r in sel]),
        })
    display(pd.DataFrame(rows).set_index('Cấu hình').round(4))

# Ma trận nhầm lẫn gộp ba lần lặp, mức 10% nhãn
if summary['runs']:
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
    for ax, cfg in zip(axes, C.MAIN_CONFIGS):
        sel = [r for r in summary['runs'] if r['config'] == cfg and r['label_pct'] == 10]
        if not sel:
            ax.axis('off'); continue
        cm = np.sum([np.asarray(r['test_confusion']) for r in sel], axis=0)
        norm = cm / cm.sum(axis=1, keepdims=True)
        ax.imshow(norm, cmap='Blues', vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f'{cm[i,j]}\n({norm[i,j]*100:.1f}%)', ha='center', va='center',
                        fontsize=9, color='white' if norm[i, j] > 0.5 else 'black')
        ax.set_xticks([0, 1], ['Animal', 'Vehicle'])
        ax.set_yticks([0, 1], ['Animal', 'Vehicle'])
        ax.set_xlabel('Dự đoán'); ax.set_ylabel('Nhãn thật'); ax.set_title(cfg)
    fig.suptitle('Ma trận nhầm lẫn trên D_test, mức 10% nhãn (gộp 3 lần lặp)', y=1.05)
    plt.tight_layout(); plt.show()

In [ ]:
# Chênh lệch ghép cặp Δ_s = Macro-F1_s(M8) − Macro-F1_s(E1), đơn vị điểm phần trăm
if summary['paired_delta']:
    rows = []
    for key, d in sorted(summary['paired_delta'].items()):
        rows.append({'So sánh': key, 'các Δ_s (điểm %)': [round(v, 3) for v in d['deltas_pp']],
                     'trung bình': round(d['mean'], 3), 'độ lệch chuẩn': round(d['std'], 3),
                     'số lần dương': f"{sum(1 for v in d['deltas_pp'] if v > 0)}/{len(d['deltas_pp'])}"})
    display(pd.DataFrame(rows))
else:
    print('Chưa có dữ liệu ghép cặp.')

## 8. Chi phí suy luận và độ trễ

Mô hình suy luận của E1 và E2 **giống hệt nhau**: chỉ encoder + Superclass Head. Rotation
Head chỉ tồn tại khi huấn luyện. Độ trễ đo với batch = 1, FP32, `eval()` +
`inference_mode()`, khởi động 50 lượt rồi đo 1.000 lượt × 3 phiên.

In [ ]:
bench_path = ROOT / 'results' / 'inference_benchmark.json'
if bench_path.exists():
    ib = json.loads(bench_path.read_text(encoding='utf-8'))
    p = ib['parameters']
    display(pd.DataFrame([
        {'Thành phần': 'Tham số encoder', 'Giá trị': f"{p['encoder']:,}"},
        {'Thành phần': 'Tham số Superclass Head', 'Giá trị': f"{p['super_head']:,}"},
        {'Thành phần': 'Tham số Rotation Head', 'Giá trị': f"{p['rotation_head']:,}"},
        {'Thành phần': 'Tổng khi suy luận', 'Giá trị': f"{p['inference_total']:,}"},
        {'Thành phần': 'Tổng khi huấn luyện', 'Giá trị': f"{p['training_total']:,}"},
        {'Thành phần': 'FP32 suy luận', 'Giá trị': f"{p['inference_fp32_mib']:.2f} MiB"},
        {'Thành phần': 'Tệp trọng số suy luận (đo thực tế)',
         'Giá trị': f"{p.get('inference_file_mib_measured', p['inference_file_mib']):.2f} MiB"},
    ]))

    lat = []
    for label, key in (('CPU (4 luồng)', 'latency_cpu'), ('GPU RTX 5070 Laptop', 'latency_gpu')):
        d = ib.get(key) or {}
        if not d:
            continue
        lat.append({'Điều kiện': label, 'mean (ms)': round(d['mean_ms'], 3),
                    'median (ms)': round(d['median_ms'], 3), 'p95 (ms)': round(d['p95_ms'], 3),
                    'số phiên': d['sessions'], 'lượt/phiên': d['iters_per_session']})
    display(pd.DataFrame(lat))
    print('Thời gian đọc ảnh + tiền xử lý (đo riêng):',
          round(ib['preprocess_ms_per_image'], 3), 'ms/ảnh')
else:
    print('Chưa có results/inference_benchmark.json — hãy chạy scripts/run_inference_benchmark.py')

## 9. Kết luận tự động từ kết quả

Phần này sinh kết luận **từ chính số liệu**, không viết cứng, để tránh việc kết luận
đi trước kết quả.

In [ ]:
sys.path.insert(0, str(ROOT / 'scripts'))
from make_report_data import _delta_stats, _finding_sentence, _data_source_sentence, _cost_sentence

if summary['runs']:
    print('C1 — Hiệu quả phân loại')
    print('  ', _finding_sentence(summary, tex=False))
    print()
    print('C2 — Vai trò của dữ liệu cho nhiệm vụ phụ')
    print('  ', _data_source_sentence(summary, tex=False))
    print()
    print('C3 — Chi phí huấn luyện')
    print('  ', _cost_sentence(summary, tex=False) or '(chưa đủ dữ liệu)')
else:
    print('Chưa có kết quả.')

In [ ]:
# Biểu đồ Macro-F1 theo mức nhãn
if summary['aggregate']:
    pcts = sorted({int(k.split('_')[1]) for k in summary['aggregate']})
    x = np.arange(len(pcts)); w = 0.26
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    for i, cfg in enumerate(C.MAIN_CONFIGS):
        m, s = [], []
        for pct in pcts:
            a = summary['aggregate'].get(f'{cfg}_{pct}')
            m.append(a['test_macro_f1']['mean'] if a else np.nan)
            s.append(a['test_macro_f1']['std'] if a else 0)
        ax.bar(x + (i - 1) * w, m, w, yerr=s, capsize=4, label=cfg)
    ax.set_xticks(x, [f'{p}%' for p in pcts])
    ax.set_xlabel('Tỷ lệ nhãn huấn luyện'); ax.set_ylabel('Macro-F1 trên D_test')
    ax.axhline(0.60, ls=':', c='gray'); ax.text(0.02, 0.62, 'đoán hết Animal = 0,60', fontsize=8)
    ax.set_ylim(0, 1.05); ax.legend(title='Cấu hình'); ax.grid(alpha=0.3)
    ax.set_title('Macro-F1 theo ngân sách nhãn (mean ± std, 3 lần lặp)')
    plt.tight_layout(); plt.show()

# Đường học tại mức 10%
if summary['runs']:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    for cfg in C.MAIN_CONFIGS:
        sel = [r for r in runs.values() if r['spec']['config'] == cfg and r['spec']['label_pct'] == 10]
        if not sel:
            continue
        curves = np.array([[h['val_macro_f1'] for h in r['history']] for r in sel])
        curves = curves[:, :curves.shape[1]]
        axes[0].plot(curves.mean(axis=0), label=cfg)
        curves2 = np.array([[h['val_sup_loss'] for h in r['history']] for r in sel])
        axes[1].plot(curves2.mean(axis=0), label=cfg)
    axes[0].set_title('Macro-F1 validation (10% nhãn)'); axes[0].set_xlabel('Epoch')
    axes[1].set_title('Loss phân loại validation (10% nhãn)'); axes[1].set_xlabel('Epoch')
    for ax in axes:
        ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

---

## 10. Mở rộng contrastive: NT-Xent và SupCon (E3–E6)

Phần này triển khai bảng 5 của đề cương. Hai hàm mất mát được cài đúng theo tài liệu gốc:

$$L_{\mathrm{NT\text{-}Xent}} = \frac{1}{2N}\sum_i -\log\frac{\exp(s_{i,j(i)}/\tau)}{\sum_{k\neq i}\exp(s_{i,k}/\tau)}$$

$$L_{\mathrm{SupCon}} = \sum_i \frac{-1}{|A(i)|}\sum_{a\in A(i)}\log\frac{\exp(s_{i,a}/\tau)}{\sum_{k\neq i}\exp(s_{i,k}/\tau)},\quadA(i)=\{a\neq i: \tilde y_a = \tilde y_i\}$$

**Ràng buộc bắt buộc:** SupCon chỉ dùng $D_L$ (không đọc nhãn thật của $D_U$);
NT-Xent dùng $D_{train}$ vì không cần nhãn; Projection Head **chỉ** dùng khi huấn luyện
nên số tham số suy luận không đổi.

In [ ]:
from m8.contrastive import contrastive_loss, nt_xent_loss, supcon_loss, ProjectionHead

# --- ví dụ tính tay cho NT-Xent ---
z = torch.tensor([[1., 0.], [0., 1.], [1., 0.], [0., 1.]])
import math
got = nt_xent_loss(z, temperature=0.5).item()
expect = math.log(math.exp(2.0) + 2.0) - 2.0
print(f'NT-Xent: nhận {got:.6f} | công thức tay {expect:.6f} -> '
      f"{'ĐẠT' if abs(got - expect) < 1e-6 else 'SAI'}")

# --- ví dụ tính tay cho SupCon ---
z2 = torch.tensor([[1., 0.], [.9, .1], [0., 1.], [.1, .9]])
lab = torch.tensor([0, 0, 1, 1])
l_sup = supcon_loss(z2, lab, temperature=0.5)
zn = torch.nn.functional.normalize(z2, dim=1); sim = zn @ zn.t() / 0.5
manual = 0.0
for i in range(4):
    others = [k for k in range(4) if k != i]
    lse = torch.logsumexp(sim[i, others], dim=0)
    pos = [k for k in others if lab[k] == lab[i]]
    manual += float((-torch.stack([sim[i, p] for p in pos]) + lse).mean())
manual /= 4
print(f'SupCon : nhận {l_sup.item():.6f} | công thức tay {manual:.6f} -> '
      f"{'ĐẠT' if abs(l_sup.item() - manual) < 1e-5 else 'SAI'}")
print('SupCon không có positive pair ->', supcon_loss(z2, torch.arange(4)).item(), '(phải là 0)')

In [ ]:
# --- Projection Head không ảnh hưởng suy luận ---
m_plain = build_model(with_rotation_head=True, seed=0)
m_proj = build_model(with_rotation_head=True, proj_dim=C.PROJ_DIM, seed=0)
s_plain, s_proj = model_summary(m_plain), model_summary(m_proj)
print('không có Projection Head: suy luận', f"{s_plain['inference_total']:,}", 'tham số')
print('có Projection Head      : suy luận', f"{s_proj['inference_total']:,}", 'tham số |',
      'Projection Head', f"{s_proj.get('projection_head', 0):,}", 'tham số')
assert s_proj['inference_total'] == s_plain['inference_total'] == 11_169_858
print('=> số tham số suy luận KHÔNG đổi: ĐẠT')

In [ ]:
# --- hai view contrastive và dataset không mang nhãn ---
from m8.data import ContrastiveViewDataset

idx_sup = bench.dl(42, 0.10)
ds_sup = ContrastiveViewDataset(train_base, idx_sup, return_superclass=True)
a, b, y, _ = ds_sup[0]
print('SupCon  : view1', tuple(a.shape), '| view2', tuple(b.shape), '| nhãn', y,
      '| hai view khác nhau:', not torch.allclose(a, b))

ds_unsup = ContrastiveViewDataset(train_base, bench.train_idx, return_superclass=False)
_, _, y2, _ = ds_unsup[0]
print('NT-Xent : nhãn trả về', y2, '(= -1 nghĩa là dataset không mang nhãn -> không rò rỉ)')

fig, axes = plt.subplots(1, 4, figsize=(10, 2.7))
def _disp(t):
    return np.clip(t.detach().numpy().transpose(1, 2, 0) * 0.5 + 0.5, 0, 1)
for ax, (im, title) in zip(axes, [(np.asarray(train_base[int(idx_sup[0])][0]), 'ảnh gốc'),
                                  (_disp(a), 'contrastive view 1'),
                                  (_disp(b), 'contrastive view 2')]):
    ax.imshow(im); ax.set_title(title, fontsize=9); ax.axis('off')
from m8.data import RotationViewDataset as _RVD
import numpy as np
axes[3].imshow(np.rot90(np.asarray(train_base[int(idx_sup[0])][0]), k=1))
axes[3].set_title('nhánh xoay k=1', fontsize=9); axes[3].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# --- kết quả khối mở rộng đã chạy (nếu có) ---
from m8.config import CHECKPOINT_EXT_DIR
from m8.experiments import load_all_runs_multi, summarize_runs as _summ

runs_ext = load_all_runs(CHECKPOINT_EXT_DIR)
print('số lượt mở rộng đã có:', len(runs_ext))
if runs_ext:
    summary_all = _summ(load_all_runs_multi(ROOT / 'results' / 'checkpoints', CHECKPOINT_EXT_DIR))
    rows = []
    for cfg in ('E1', 'E2-L', 'E2', 'E3', 'E4', 'E5', 'E6'):
        for pct in sorted({r['label_pct'] for r in summary_all['runs']}):
            a = summary_all['aggregate'].get(f'{cfg}_{pct}')
            d = summary_all['paired_delta'].get(f'{cfg}_minus_E1_{pct}')
            if not a:
                continue
            rows.append({'Cấu hình': cfg, 'Nhãn': f'{pct}%',
                         'Test Macro-F1': f"{a['test_macro_f1']['mean']:.4f} ± {a['test_macro_f1']['std']:.4f}",
                         'Δ so E1 (điểm %)': ('—' if cfg == 'E1' or not d else f"{d['mean']:+.2f}"),
                         'số lần dương': ('—' if cfg == 'E1' or not d else
                                         f"{sum(1 for v in d['deltas_pp'] if v > 0)}/{d['n']}")})
    display(pd.DataFrame(rows))
else:
    print('Chưa có kết quả mở rộng — chạy scripts/run_extensions.py trước.')

In [ ]:
# --- khảo sát λ_c: CHỈ dùng validation, không đụng tới test ---
import json
sweep_path = ROOT / 'results' / 'lambda_c_sweep.json'
if sweep_path.exists():
    sw = json.loads(sweep_path.read_text(encoding='utf-8'))
    rec = pd.DataFrame(sw['records'])[['config', 'kind', 'source', 'label_pct',
                                       'lambda_c', 'best_val_macro_f1', 'best_epoch']]
    display(rec.sort_values(['config', 'lambda_c']).reset_index(drop=True))
    print('λ_c được chọn theo validation:', json.dumps(sw['chosen'], ensure_ascii=False))
else:
    print('Chưa có khảo sát — chạy scripts/sweep_lambda_c.py')

**Kết luận của phần mở rộng** được sinh tự động từ số liệu (xem ô dưới), để tránh việc
kết luận đi trước kết quả.

In [ ]:
from make_report_data import _extension_text

if runs_ext:
    analysis, conclusion = _extension_text(summary_all)
    print('PHÂN TÍCH:\n', analysis.replace('\\cfg{', '').replace('}', '').replace('\\%', '%'))
    print()
    print('KẾT LUẬN:\n', conclusion.replace('\\cfg{', '').replace('}', '')
          .replace('\\%', '%').replace('\\textbf{', '').replace('\\,', ' '))
else:
    print('Chưa có kết quả mở rộng.')

## 11. Ghi chú về tái lập và giới hạn

**Tái lập.** Seed chia split 2026; seed lần lặp 42, 43, 44 dùng cho khởi tạo trọng số, thứ tự
dữ liệu, augmentation và bộ sinh số riêng của nhánh phụ. Chỉ mục đầy đủ của mọi tập nằm ở
`data/artifacts/splits.json`.

**Giới hạn cần nhớ khi đọc kết quả.**

* Ba lần lặp chỉ cho bằng chứng ban đầu về độ ổn định, **không** đủ để kết luận về ý nghĩa
  thống kê.
* Phép chia train/validation/test là cố định, nên phương sai do chính phép chia gây ra không
  được ước lượng.
* BatchNorm được dùng chung cho hai nhánh, nên so sánh E2-L/E2 với E1 trộn lẫn tác dụng của
  nhiệm vụ phụ với thay đổi thống kê chuẩn hóa; **không** nên diễn giải là phép tách hoàn toàn.
* Kết quả độ trễ trên CPU/GPU của máy tính **không** phải benchmark trên thiết bị IoT đích.
* Không dùng Early Stopping, nên các lượt chạy có thể đã quá khớp ở cuối 100 epoch.